# 4장 복습 — 풀이

## 0. 준비
**이 셀부터** 실행하자. 채점 도우미 `check`와, 4장에서 만든 함수들의 **올바른 버전**이 들어 있다. (Part C, D에서 쓴다)

In [ ]:
import sys, os
sys.path.append(os.pardir)
import numpy as np
import matplotlib.pyplot as plt

def check(got, want, name='답'):
    if got is None:
        raise AssertionError(f'{name}: 아직 안 채웠어요 (None)')
    if isinstance(want, (tuple, str, bool)):
        ok = got == want
    else:
        ok = np.shape(got) == np.shape(want) and np.allclose(got, want)
    mark = '✅' if ok else '❌'
    print(f'{name} = {got!r}   (정답 {want!r}) {mark}')
    assert ok, f'{name} 이(가) 틀렸어요'

# ---- 4장에서 만든 도구 (올바른 버전) ----
def numerical_diff(f, x):
    h = 1e-4
    return (f(x + h) - f(x - h)) / (2 * h)

def tangent_line(f, x):
    d = numerical_diff(f, x)
    b = f(x) - d * x
    return lambda t: d * t + b

def numerical_gradient(f, x):          # 점 하나 (1차원)
    h = 1e-4
    grad = np.zeros_like(x)
    for i in range(x.size):
        tmp_val = x[i]
        x[i] = tmp_val + h
        fxh1 = f(x)
        x[i] = tmp_val - h
        fxh2 = f(x)
        grad[i] = (fxh1 - fxh2) / (2 * h)
        x[i] = tmp_val
    return grad

def numerical_gradient_2d(f, X):       # 점 여러 개 (행 = 점)
    if X.ndim == 1:
        return numerical_gradient(f, X)
    grad = np.zeros_like(X)
    for idx, x in enumerate(X):
        grad[idx] = numerical_gradient(f, x)
    return grad

def function_1(x):
    return 0.01 * x ** 2 + 0.1 * x

def function_2(x):
    return np.sum(x ** 2)

print('준비 완료')

# Part A. 인덱싱과 모양
## A-1. 🔮 `[1]` vs `[[1]]` vs `[:, 1]`
```python
M = np.arange(12).reshape(3, 4)
# [[ 0  1  2  3]
#  [ 4  5  6  7]
#  [ 8  9 10 11]]
```
**실행하지 말고** 모양을 튜플로 적어 보자.
- `p1`: `M[1].shape`
- `p2`: `M[[1]].shape`
- `p3`: `M[[0, 2]].shape`
- `p4`: `M[:, 1].shape`  — `:`는 "모든 행", 그중 1번 열
- `p5`: `M[:, [1]].shape`  — 열 쪽에 **리스트**를 넣으면?

> 규칙: 숫자 하나로 꺼내면 그 축이 **사라지고**, 리스트로 꺼내면 그 축이 **남는다**(개수 = 리스트 길이).

In [ ]:
p1 = (4,)      # 숫자 1 → 행 축이 사라짐 → 1차원
p2 = (1, 4)    # 리스트 [1] → 행 축이 남음 (1개)
p3 = (2, 4)    # 리스트 [0, 2] → 행 2개
p4 = (3,)      # 열 쪽에 숫자 1 → 열 축이 사라짐 → 1차원 (3개 행의 1번 값)
p5 = (3, 1)    # 열 쪽에 리스트 [1] → 열 축이 남음 → 세로 한 줄짜리 2차원

In [ ]:
# 확인
M = np.arange(12).reshape(3, 4)
print(M[1].shape, M[[1]].shape, M[[0, 2]].shape, M[:, 1].shape, M[:, [1]].shape)
check(p1, (4,), 'p1'); check(p2, (1, 4), 'p2'); check(p3, (2, 4), 'p3')
check(p4, (3,), 'p4'); check(p5, (3, 1), 'p5')
print('통과!')

## A-2. 🔮 짝지어 꺼내기 vs 행 묶음 꺼내기
```python
P = np.array([[0.1, 0.2, 0.7],
              [0.5, 0.4, 0.1],
              [0.3, 0.3, 0.4],
              [0.2, 0.6, 0.2]])
t = np.array([2, 0, 2, 1])
```
**실행하지 말고** 적어 보자.
- `pick_a`: `P[np.arange(4), t]` 의 값 (리스트)
- `pick_b`: `P[[0, 1], [2, 0]]` 의 값 (리스트) — 대괄호 **하나** 안에 목록 두 개
- `pick_c`: `P[[0, 1]].shape` — 목록 **하나**

> 대괄호 하나 안에 `행 목록, 열 목록`을 쉼표로 넣으면 **같은 순서끼리 짝**을 짓는다: (행0, 열0), (행1, 열1), ...

In [ ]:
pick_a = [0.7, 0.5, 0.4, 0.6]   # (0,2) (1,0) (2,2) (3,1)
pick_b = [0.7, 0.5]             # (0,2) (1,0) — np.arange(2) 대신 [0, 1]을 직접 쓴 것과 같다
pick_c = (2, 3)                 # 목록 하나 → 0번, 1번 '행'을 통째로

In [ ]:
# 확인
P = np.array([[0.1, 0.2, 0.7],
              [0.5, 0.4, 0.1],
              [0.3, 0.3, 0.4],
              [0.2, 0.6, 0.2]])
t = np.array([2, 0, 2, 1])
print(P[np.arange(4), t], P[[0, 1], [2, 0]], P[[0, 1]].shape)
check(pick_a, [0.7, 0.5, 0.4, 0.6], 'pick_a')
check(pick_b, [0.7, 0.5], 'pick_b')
check(pick_c, (2, 3), 'pick_c')
print('통과!')

### 🔁 이어서: 배치 손실을 한 줄로
위의 `P`(데이터 4개의 확률)와 정답 숫자 `t`로
- `losses`: 데이터마다의 손실 `-log(정답 자리 확률 + 1e-7)` — 배열 그대로 (4,)
- `mean_loss`: 그 평균 (= 배치 교차 엔트로피)
- `acc`: 정확도 — 데이터마다 `argmax`한 예측과 `t`를 비교

In [ ]:
losses = -np.log(P[np.arange(4), t] + 1e-7)   # (4,)
mean_loss = np.mean(losses)                    # = np.sum(losses) / 4
acc = np.sum(np.argmax(P, axis=1) == t) / 4    # 예측 [2, 0, 2, 1] → 모두 맞음

In [ ]:
# 확인
print(losses)
check(losses.shape, (4,), 'losses.shape')
check(mean_loss, -np.mean(np.log(np.array([0.7, 0.5, 0.4, 0.6]) + 1e-7)), 'mean_loss')
check(acc, 1.0, 'acc')
print('통과!')

## A-3. 🔮 `axis` — 어느 쪽을 눌러 없애나
위의 `M` (3, 4)로 **실행하지 말고** 적어 보자. (리스트로)
- `s0`: `M.sum(axis=0)`
- `s1`: `M.sum(axis=1)`
- `a0`: `np.argmax(M, axis=0)`

> 요령: `axis=0`이면 0번 축(행)을 **눌러 없앤다** → 위아래로 더함 → 결과 길이 = 열 개수(4).
> `axis=1`이면 1번 축(열)을 없앤다 → 한 행 안에서 더함 → 결과 길이 = 행 개수(3). 신경망의 "데이터마다"는 보통 `axis=1`.

In [ ]:
s0 = [12, 15, 18, 21]   # 열마다 위아래로: 0+4+8, 1+5+9, ...
s1 = [6, 22, 38]        # 행마다 옆으로: 0+1+2+3, 4+5+6+7, 8+9+10+11
a0 = [2, 2, 2, 2]       # 열마다 가장 큰 값은 모두 맨 아래 행(2)

In [ ]:
# 확인
print(M.sum(axis=0), M.sum(axis=1), np.argmax(M, axis=0))
check(s0, [12, 15, 18, 21], 's0'); check(s1, [6, 22, 38], 's1'); check(a0, [2, 2, 2, 2], 'a0')
print('통과!')

## A-4. reshape는 언제 필요한가
**"행 개수 = 데이터 개수"**로 세는 코드가 있을 때만 필요하다.

### 🔮 먼저 예측하기
- `n_wrong`: `np.zeros(10).shape[0]` — 데이터 **1개**(길이 10)인데, 행 개수로 세면?

### 그다음 만들기
- `as_batch(y)`: `y`가 1차원이면 `(1, 크기)`로 바꿔서 돌려주고, 2차원이면 그대로 돌려준다
- `count_data(y)`: `as_batch`를 거친 뒤 행 개수를 돌려준다

In [ ]:
n_wrong = 10   # 1차원이라 shape[0]이 '원소 개수'가 돼 버린다 → 데이터 10개로 착각

def as_batch(y):
    if y.ndim == 1:
        return y.reshape(1, y.size)   # 1행짜리 묶음으로
    return y

def count_data(y):
    return as_batch(y).shape[0]

In [ ]:
# 확인
check(n_wrong, 10, 'n_wrong')
check(as_batch(np.zeros(10)).shape, (1, 10), 'as_batch(1차원)')
check(as_batch(np.zeros((5, 10))).shape, (5, 10), 'as_batch(2차원)')
check(count_data(np.zeros(10)), 1, 'count_data(1차원)')
check(count_data(np.zeros((5, 10))), 5, 'count_data(2차원)')
print('통과!')

## A-5. ⚠️🔮 reshape는 원본을 바꾸지 않는다
```python
a = np.arange(6)
a.reshape(2, 3)
print(a.shape)
```
- `p_shape`: 출력되는 모양은? (튜플)

In [ ]:
p_shape = (6,)   # reshape는 '새 배열'을 돌려줄 뿐. 담지 않으면 a는 그대로 → a = a.reshape(2, 3)

In [ ]:
# 확인
a = np.arange(6)
a.reshape(2, 3)
print(a.shape)
check(p_shape, (6,), 'p_shape')
print('통과! 바꾼 결과를 쓰려면 a = a.reshape(2, 3) 처럼 다시 담아야 한다.')

# Part B. 에러 찾기 — 내가 했던 실수
아래 코드에는 4장을 풀면서 **실제로 했던 실수**가 그대로 들어 있다. 실행해서 확인 셀이 실패하는 걸 본 다음, 코드를 **고쳐서** 통과시키자.
(함수 이름에 `my_`를 붙여서, 0번 준비 셀의 올바른 함수와 섞이지 않게 했다)

먼저 데이터 셀을 실행하자.

In [ ]:
# 책 예제 데이터 (실행만)
t1 = np.array([0, 0, 1, 0, 0, 0, 0, 0, 0, 0])
y1 = np.array([0.1, 0.05, 0.6, 0.0, 0.05, 0.1, 0.0, 0.1, 0.0, 0.0])
y2 = np.array([0.1, 0.05, 0.1, 0.0, 0.05, 0.1, 0.0, 0.6, 0.0, 0.0])
Yb = np.array([y1, y2])        # (2, 10) 배치

## B-1. 평균 제곱 오차 (버그 2개)

In [ ]:
def my_mse(y, t):
    loss = 0.5 * np.sum((t - y) * (t - y))   # ① 0.5 빠짐  ② return 빠짐  (+ 내장 함수 이름 sum 피하기)
    return loss

In [ ]:
# 확인
check(my_mse(y1, t1), 0.0975, 'my_mse(y1)')
print('통과!')

## B-2. 수치 미분 (버그 1개)

In [ ]:
def my_numerical_diff(f, x):
    h = 1e-4
    return (f(x + h) - f(x - h)) / (2 * h)   # 괄호 없으면 2로 나누고 h를 '곱함'

In [ ]:
# 확인: x**2 의 x=3 기울기 = 6
check(my_numerical_diff(lambda x: x ** 2, 3.0), 6.0, 'my_numerical_diff')
print('통과!')

## B-3. 배치용 교차 엔트로피 — 원-핫 (버그 1개, 두 줄에 같은 실수)

In [ ]:
def my_cee_batch(y, t):
    if y.ndim == 1:
        t = t.reshape(1, t.size)   # reshape 결과를 다시 담기
        y = y.reshape(1, y.size)
    batch_size = y.shape[0]
    return -np.sum(t * np.log(y + 1e-7)) / batch_size

In [ ]:
# 확인
check(my_cee_batch(y1, t1), 0.510825457099338, 'my_cee_batch(y1)')
check(my_cee_batch(Yb, np.array([t1, t1])), 1.406704775046942, 'my_cee_batch(Yb)')
print('통과!')

## B-4. 배치용 교차 엔트로피 — 정답 레이블 (버그 2개)

In [ ]:
def my_cee_label(y, t):
    if y.ndim == 1:
        y = y.reshape(1, y.size)
        t = t.reshape(1, t.size)
    batch_size = y.shape[0]
    result = y[np.arange(batch_size), t]                # y.size(원소 전체) → batch_size(행 개수)
    return -np.sum(np.log(result + 1e-7)) / batch_size  # log가 음수라서 앞에 -

In [ ]:
# 확인
check(my_cee_label(Yb, np.array([2, 2])), 1.406704775046942, 'my_cee_label(Yb)')
print('통과!')

## B-5. 접선 (버그 1개)

In [ ]:
def my_tangent_line(f, x):
    diff_f = numerical_diff(f, x)
    return lambda t: diff_f * (t - x) + f(x)   # diff_f는 숫자 → 곱하기 * 필요 (diff_f(...)는 '함수 부르기')

In [ ]:
# 확인
check(my_tangent_line(function_1, 5)(0), -0.25, 'my_tangent_line(f1, 5)(0)')
print('통과!')

## B-6. 기울기 (버그 2개)

In [ ]:
def my_numerical_gradient(f, x):
    h = 1e-4
    grad = np.zeros_like(x)
    for i in range(x.size):
        tmp_val = x[i]
        x[i] = tmp_val + h
        fpx = f(x)
        x[i] = tmp_val - h
        fmx = f(x)
        grad[i] = (fpx - fmx) / (2 * h)   # ① id(내장 함수) → i   ② 괄호
        x[i] = tmp_val
    return grad

In [ ]:
# 확인
check(my_numerical_gradient(function_2, np.array([3.0, 4.0])), np.array([6.0, 8.0]), 'my_numerical_gradient')
print('통과!')

## B-7. 여러 점의 기울기 (버그 1개)

In [ ]:
def my_numerical_gradient_2d(f, X):
    if X.ndim == 1:
        return numerical_gradient(f, X)
    grad = np.zeros_like(X)
    for idx, x in enumerate(X):
        grad[idx] = numerical_gradient(f, x)   # '어떤 함수(f)'를 '어떤 점(x)'에서 — 둘 다 넘기기
    return grad

In [ ]:
# 확인
Pts = np.array([[3.0, 4.0], [0.0, 2.0], [3.0, 0.0]])
check(my_numerical_gradient_2d(function_2, Pts), np.array([[6.0, 8.0], [0.0, 4.0], [6.0, 0.0]]), 'my_numerical_gradient_2d')
print('통과!')

## B-8. 정답 숫자 꺼내기 (버그 1개)

In [ ]:
T5 = np.array([[0, 0, 1],
               [1, 0, 0],
               [0, 1, 0],
               [0, 0, 1]])
shape_guess = (4, 3)   # T5의 모양을 예측해 둔 메모

labels5 = np.argmax(T5, axis=1)   # 모양 메모(튜플)가 아니라 실제 데이터 T5를 넣어야 한다

In [ ]:
# 확인
check(labels5, np.array([2, 0, 1, 2]), 'labels5')
print('통과!')

## B-9. 화살표 방향 (버그 1개)
점 두 개에서 **최솟값 (0, 0) 쪽**을 가리키는 화살표의 가로·세로 성분 `U`, `V`를 구하려고 한다.

In [ ]:
pts2 = np.array([[1.0, 1.0],
                 [-1.0, 2.0]])
g2 = numerical_gradient_2d(function_2, pts2)
U = -g2[:, 0]   # 기울기는 오르막 방향 → 마이너스를 붙여야 내리막(최솟값 쪽)
V = -g2[:, 1]

In [ ]:
# 확인: (1, 1)에서는 왼쪽 아래(-, -)로, (-1, 2)에서는 오른쪽 아래(+, -)로 가야 원점이다
check(U, np.array([-2.0, 2.0]), 'U')
check(V, np.array([-2.0, -4.0]), 'V')
print('통과!')

# Part C. `meshgrid`와 점 쌓기
## C-1. 🔮 meshgrid 결과 예측하기
```python
x0 = np.array([0, 1, 2])   # 가로 눈금 3개
x1 = np.array([5, 6])      # 세로 눈금 2개
X, Y = np.meshgrid(x0, x1)
```
**실행하지 말고** 적어 보자.
- `mg_shape`: `X.shape` (튜플)
- `mg_X`: `X`의 값 (리스트 안에 리스트)
- `mg_Y`: `Y`의 값

> 기억: 표 모양 = 좌표평면 모양 = **(x1 개수, x0 개수)**. `X`는 x0 줄을 아래로 복사, `Y`는 x1 값을 옆으로 늘어놓기.

In [ ]:
mg_shape = (2, 3)
mg_X = [[0, 1, 2],
        [0, 1, 2]]   # x0 줄을 x1 개수(2)만큼 아래로 복사
mg_Y = [[5, 5, 5],
        [6, 6, 6]]   # x1 값을 x0 개수(3)만큼 옆으로

In [ ]:
# 확인
x0 = np.array([0, 1, 2])
x1 = np.array([5, 6])
X, Y = np.meshgrid(x0, x1)
print(X); print(Y)
check(mg_shape, (2, 3), 'mg_shape')
check(mg_X, X, 'mg_X')
check(mg_Y, Y, 'mg_Y')
print('통과!')

## C-2. 🔮 펴기와 점 번호
- `flat_X`, `flat_Y`: `X.flatten()`, `Y.flatten()`의 값 (리스트)
- `where_25`: 펼친 뒤 점 **(2, 5)**는 몇 번 칸일까? (0부터 셈)

In [ ]:
flat_X = [0, 1, 2, 0, 1, 2]   # 위 행부터 차례로
flat_Y = [5, 5, 5, 6, 6, 6]
where_25 = 2                  # flat_X[2] = 2, flat_Y[2] = 5

In [ ]:
# 확인
print(X.flatten(), Y.flatten())
check(flat_X, X.flatten(), 'flat_X')
check(flat_Y, Y.flatten(), 'flat_Y')
check(where_25, 2, 'where_25')
print('통과!')

## C-3. 🔁 점을 행으로 쌓기
- `pts`: 펼친 `X`, `Y`로 **점 하나 = 행 하나**인 배열 `(6, 2)`를 만들자
- `back`: `pts`를 다시 `(2, 6)` 모양(0행 = 모든 x0, 1행 = 모든 x1)으로 바꾸자

In [ ]:
Xf = X.flatten()
Yf = Y.flatten()

pts = np.array([Xf, Yf]).T   # (2, 6) 쌓기 → .T → (6, 2)
back = pts.T                 # 다시 (2, 6): back[0] = 모든 x0, back[1] = 모든 x1

In [ ]:
# 확인
print(pts)
check(pts.shape, (6, 2), 'pts.shape')
check(pts[4], np.array([1, 6]), 'pts[4]')
check(back.shape, (2, 6), 'back.shape')
check(back[0], Xf, 'back[0]')
print('통과!')

## C-4. 🔁 격자 전체의 함수 값 — 두 가지 방법
$f(x_0, x_1) = x_0^2 + x_1^2$ 의 값을 6개 점 모두에서 구하자.

1. `Z_grid`: **2차원 `X`, `Y`를 그대로** 써서 한 줄로 (원소끼리 계산이라 반복문이 필요 없다)
2. `Z_loop`: `pts`의 **행마다** `function_2`를 불러서 (리스트 컴프리헨션 `[... for p in pts]` 또는 반복문) → `np.array`로
3. `Z_back`: `Z_loop`(길이 6)를 `X`와 같은 모양으로 되돌리기 — `reshape`

In [ ]:
Z_grid = X ** 2 + Y ** 2                        # (2, 3) — 격자 모양 그대로
Z_loop = np.array([function_2(p) for p in pts])  # (6,) — 점마다 하나
Z_back = Z_loop.reshape(X.shape)                 # (6,) → (2, 3)

In [ ]:
# 확인
print(Z_grid)
check(Z_grid.shape, (2, 3), 'Z_grid.shape')
check(Z_loop.shape, (6,), 'Z_loop.shape')
check(Z_back, Z_grid, 'Z_back')
print('통과! flatten ↔ reshape(X.shape)는 서로 되돌리는 관계다.')

## C-5. ⭐ 같은 일을 하는 다른 함수 (책 범위 밖)
`np.array([Xf, Yf]).T` 대신 `np.column_stack([Xf, Yf])`(세로 줄로 나란히 붙이기)도 같은 `(6, 2)`를 만든다. 실행만 해 보자.

In [ ]:
print(np.column_stack([Xf, Yf]))
print('같은가?', np.array_equal(np.column_stack([Xf, Yf]), pts))

# Part D. 행렬로 그래프 그리기
나중에 최적화를 공부할 때 계속 쓰는 그림들이다. 핵심은 **"행렬의 한 행(또는 한 열) = 그래프의 한 줄(또는 점들)"**.

## D-1. 🔁 접선 여러 개를 한 그림에
`function_1`의 `x = 5, 10, 15`에서의 접선 3개를 그린다.
1. `slopes`: 세 점의 기울기 (`numerical_diff`) — 배열 (3,)
2. `lines`: 접선 3개의 값을 **행으로 쌓은 행렬** — 행 하나 = 접선 하나 (`tangent_line(function_1, p)(xs)`)
3. `for row in lines:`로 한 행씩 `plt.plot(xs, row)` + 원래 함수도 그리기

### 🔮 먼저 예측하기
`xs = np.arange(0.0, 20.0, 0.1)`(200개)일 때 `lines.shape`은?

In [ ]:
predict_lines = (3, 200)   # 접선 3개 × x값 200개

In [ ]:
xs = np.arange(0.0, 20.0, 0.1)
points = [5, 10, 15]

slopes = np.array([numerical_diff(function_1, p) for p in points])   # [0.2, 0.3, 0.4]
lines = np.array([tangent_line(function_1, p)(xs) for p in points])  # (3, 200)

plt.plot(xs, function_1(xs), label='f(x)', linewidth=2)
for p, row in zip(points, lines):          # zip: 두 목록을 같은 순서끼리 짝지어 꺼내기
    plt.plot(xs, row, '--', label=f'tangent at x={p}')
plt.ylim(-1, 6)
plt.legend()
plt.show()

In [ ]:
# 확인
check(predict_lines, (3, 200), 'predict_lines')
check(slopes, np.array([0.2, 0.3, 0.4]), 'slopes')
check(lines.shape, (3, 200), 'lines.shape')
check(lines[1][100], tangent_line(function_1, 10)(xs[100]), 'lines[1][100]')
print('통과!')

## D-2. 다른 함수의 기울기 화살표 — 처음부터 직접
$$g(x_0, x_1) = (x_0 - 1)^2 + x_1^2$$
그림 4-9를 이 함수로 **처음부터** 다시 그려 보자. 결과는 아래 이름으로 만들기.
- `function_g(x)`
- `Xg`, `Yg`: 펼친 격자 좌표 (눈금은 `np.arange(-2, 2.5, 0.25)`, 각 324개)
- `grad_g`: 324개 점의 기울기 `(324, 2)`
- 화살표: `plt.quiver(...)` — 최솟값 쪽을 가리키게

### 🔮 먼저 예측하기
- `center`: 화살표가 모두 모이는 점은? (튜플)

In [ ]:
center = (1, 0)   # (x0 - 1)²이 0이 되는 x0 = 1, x1² 이 0이 되는 x1 = 0

In [ ]:
def function_g(x):
    return (x[0] - 1) ** 2 + x[1] ** 2

g0 = np.arange(-2, 2.5, 0.25)
Xg, Yg = np.meshgrid(g0, g0)
Xg = Xg.flatten()
Yg = Yg.flatten()

grad_g = numerical_gradient_2d(function_g, np.array([Xg, Yg]).T)   # (324, 2)

plt.figure()
plt.quiver(Xg, Yg, -grad_g[:, 0], -grad_g[:, 1], angles='xy', color='#666666')
plt.plot(1, 0, 'ro')   # 최솟값 표시
plt.xlim([-2, 2]); plt.ylim([-2, 2])
plt.xlabel('x0'); plt.ylabel('x1')
plt.grid()
plt.show()

In [ ]:
# 확인: 손으로 미분한 값(2(x0 - 1), 2x1)과 비교
check(center, (1, 0), 'center')
check(Xg.shape, (324,), 'Xg.shape')
check(grad_g.shape, (324, 2), 'grad_g.shape')
check(grad_g, np.array([2 * (Xg - 1), 2 * Yg]).T, 'grad_g (손 계산과 비교)')
print('통과!')

## D-3. ⭐ 등고선 — 함수 값이 같은 점을 이은 선 (책 범위 밖)
지도의 등고선처럼, `plt.contour(X, Y, Z)`는 **함수 값이 같은 점들**을 선으로 잇는다.
이때는 `flatten`하지 않은 **2차원** `X`, `Y`, `Z`를 넣는다. (C-4의 `Z_grid`처럼)

아래 셀을 실행해서 등고선과 기울기 화살표를 **같이** 보자.

In [ ]:
c0 = np.arange(-2, 2.25, 0.25)
Xc, Yc = np.meshgrid(c0, c0)        # 2차원 그대로
Zc = Xc ** 2 + Yc ** 2              # 2차원 함수 값 (C-4의 방법 1)

Gc = numerical_gradient_2d(function_2, np.array([Xc.flatten(), Yc.flatten()]).T)

plt.figure(figsize=(5, 5))
plt.contour(Xc, Yc, Zc, levels=8)                     # 등고선
plt.quiver(Xc.flatten(), Yc.flatten(), -Gc[:, 0], -Gc[:, 1], angles='xy', color='#666666')
plt.gca().set_aspect('equal')                         # 가로세로 비율 같게 (원이 원으로 보이게)
plt.xlabel('x0'); plt.ylabel('x1')
plt.show()

### 생각해 보기
- `angle`: 화살표와 등고선은 서로 몇 도로 만날까? (숫자, 도 단위)

In [ ]:
angle = 90   # 기울기는 등고선과 '직각' 방향 = 가장 가파르게 내려가는 방향

In [ ]:
# 확인
check(angle, 90, 'angle')
print('통과! 등고선을 따라가면 값이 안 변하고, 직각 방향이 가장 빨리 변한다.')

## D-4. 🔁 경사 하강 경로를 등고선 위에 그리기
4장 그림 4-10을 학습률별로 비교해 보자.

`gradient_descent_history(f, init_x, lr, step_num)`
- `x = init_x.copy()`로 시작 (⚠️ 4-6 함정: copy 안 하면 init_x가 바뀐다)
- 처음 위치를 포함해서 **매 위치를** `x.copy()`로 리스트에 모은다
- `np.array(...)`로 바꿔 돌려준다 → 모양 `(step_num + 1, 2)` — 행 하나 = 한 시점의 위치

### 🔮 먼저 예측하기
- `predict_hist`: `step_num=20`일 때 결과 모양은? (튜플)

In [ ]:
predict_hist = (21, 2)   # 처음 위치 1개 + 이동 20번

In [ ]:
def gradient_descent_history(f, init_x, lr=0.1, step_num=20):
    x = init_x.copy()
    history = [x.copy()]                 # 처음 위치
    for i in range(step_num):
        x -= lr * numerical_gradient(f, x)
        history.append(x.copy())         # copy 안 하면 모든 칸이 같은 배열(마지막 값)이 된다
    return np.array(history)             # (step_num + 1, 2)

In [ ]:
# 확인
init = np.array([-3.0, 4.0])
hist = gradient_descent_history(function_2, init, lr=0.1, step_num=20)
check(predict_hist, (21, 2), 'predict_hist')
check(hist.shape, (21, 2), 'hist.shape')
check(hist[0], np.array([-3.0, 4.0]), 'hist[0]')
check(hist[-1], np.array([-3.0, 4.0]) * 0.8 ** 20, 'hist[-1]')   # x² 은 한 번에 (1 - 2*0.1) = 0.8배씩 줄어든다
check(init, np.array([-3.0, 4.0]), 'init (바뀌면 안 됨)')
print('통과!')

### 🔮 학습률 비교 — 예측 후 실행
`lr = 0.1, 0.9, 1.05`로 경로를 그린다. 실행하기 전에 예측하자.
- `predict_flip`: `lr=0.9`일 때 `x0`의 부호가 매번 `+`, `-`로 바뀔까? `True` / `False`
- `predict_diverge`: 셋 중 최솟값에서 **점점 멀어지는** 학습률은? (숫자)

> 힌트: $x^2$ 의 기울기는 $2x$ → 한 번 이동하면 $x - lr \cdot 2x = (1 - 2\,lr)\,x$. 세 경우에 $1 - 2\,lr$을 계산해 보자.

In [ ]:
predict_flip = True      # 1 - 2*0.9 = -0.8 → 매번 부호가 바뀌며 0.8배 (지그재그로 수렴)
predict_diverge = 1.05   # 1 - 2*1.05 = -1.1 → 절댓값이 1.1배씩 커짐 (발산)

In [ ]:
# 그래프 + 확인 (실행만)
d0 = np.arange(-6, 6.1, 0.1)
Xd, Yd = np.meshgrid(d0, d0)
Zd = Xd ** 2 + Yd ** 2

plt.figure(figsize=(6, 6))
plt.contour(Xd, Yd, Zd, levels=12, colors='#cccccc')
for lr in [0.1, 0.9, 1.05]:
    h = gradient_descent_history(function_2, np.array([-3.0, 4.0]), lr=lr, step_num=12)
    plt.plot(h[:, 0], h[:, 1], 'o-', markersize=3, label=f'lr={lr}')   # 0열 = x0들, 1열 = x1들
plt.plot(0, 0, 'k+', markersize=12)
plt.xlim(-6, 6); plt.ylim(-6, 6)
plt.gca().set_aspect('equal')
plt.legend()
plt.show()

h09 = gradient_descent_history(function_2, np.array([-3.0, 4.0]), lr=0.9, step_num=4)
print('lr=0.9 의 x0:', h09[:, 0])
check(predict_flip, bool(np.all(h09[:-1, 0] * h09[1:, 0] < 0)), 'predict_flip')
check(predict_diverge, 1.05, 'predict_diverge')
print('통과!')

## D-5. ⭐ 길쭉한 그릇에서의 지그재그 (6장 미리보기, 책 범위 밖)
$$f(x_0, x_1) = \frac{x_0^2}{20} + x_1^2$$
가로로 길쭉한 그릇이다. 같은 경사 하강법이어도 **방향마다 기울기 크기가 달라서** 지그재그로 움직인다.
6장(매개변수 갱신)에서 모멘텀, AdaGrad, Adam 같은 방법이 이 문제를 어떻게 푸는지 배운다. 최적화 공부의 출발점인 그림이다. (실행만)

In [ ]:
def f_long(x):
    return x[0] ** 2 / 20.0 + x[1] ** 2

L0 = np.arange(-10, 10.1, 0.1)
L1 = np.arange(-3, 3.05, 0.05)
XL, YL = np.meshgrid(L0, L1)
ZL = XL ** 2 / 20.0 + YL ** 2

hL = gradient_descent_history(f_long, np.array([-7.0, 2.0]), lr=0.9, step_num=30)

plt.figure(figsize=(8, 3.5))
plt.contour(XL, YL, ZL, levels=15, colors='#cccccc')
plt.plot(hL[:, 0], hL[:, 1], 'o-', color='red', markersize=3)
plt.plot(0, 0, 'k+', markersize=12)
plt.xlabel('x0'); plt.ylabel('x1')
plt.show()
print('30번 뒤 위치:', hL[-1])

### 생각해 보기
- `faster`: 30번 뒤 0에 더 가까워진 좌표는? `'x0'` 또는 `'x1'`

> 힌트: 한 번 이동할 때 $x_0$은 $(1 - 0.9 \times \frac{2}{20})$배, $x_1$은 $(1 - 0.9 \times 2)$배가 된다.

In [ ]:
faster = 'x1'   # x1: |−0.8|배씩 빠르게(지그재그) / x0: 0.91배씩 아주 느리게

In [ ]:
# 확인
check(faster, 'x1' if abs(hL[-1][1]) < abs(hL[-1][0]) else 'x0', 'faster')
print('통과! 한 학습률로는 두 방향을 동시에 잘 맞추기 어렵다 → 6장의 주제')

# Part E. 이름표와 상자 — 복사인가, 같은 배열인가
파이썬 변수는 값을 담는 **상자**가 아니라, 물건(배열)에 붙이는 **이름표**다.

```
b = a          →   [ 1, 2 ]  ← a, b   (상자 1개, 이름표 2개)
b = a.copy()   →   [ 1, 2 ]  ← a
                   [ 1, 2 ]  ← b      (상자 2개)
```

| 코드 | 하는 일 | 같은 상자의 다른 이름표에도 보이나? |
|---|---|---|
| `b = a` | 이름표 하나 더 붙이기 (복사 X) | — |
| `b[0] = 9`, `b -= 1`, `b += 1` | 상자를 **그 자리에서** 고침 | **보인다** |
| `b = b - 1` | 계산 결과로 **새 상자**를 만들고 이름표 `b`를 옮김 | 안 보인다 |
| `b = a.copy()` | 새 상자에 복사 | 안 보인다 |

`a is b` = "둘이 **완전히 같은 상자**인가?" (`==`는 "값이 같은가?")

대부분 🔮 **실행 전에 예측하기** 문제다. 종이에 상자와 이름표를 그려 가며 풀어 보자.

## E-1. 🔮 이름표만 하나 더
```python
a = np.array([1, 2, 3])
b = a
b[0] = 100
```
- `e1_a`: 이제 `a`의 값은? (리스트)
- `e1_is`: `a is b`는? (`True` / `False`)

In [ ]:
e1_a = [100, 2, 3]   # b = a 는 복사가 아니라 같은 상자에 이름표 b를 붙인 것 → b로 고치면 a도 바뀐다
e1_is = True         # 같은 상자

In [ ]:
# 확인
a = np.array([1, 2, 3])
b = a
b[0] = 100
print('a =', a, '  a is b =', a is b)
check(e1_a, a, 'e1_a')
check(e1_is, a is b, 'e1_is')
print('통과!')

## E-2. 🔮 `.copy()`로 상자를 따로
```python
a = np.array([1, 2, 3])
b = a.copy()
b[0] = 100
```
- `e2_a`: `a`의 값은?
- `e2_is`: `a is b`는?
- `e2_eq`: 처음(`b[0] = 100` **하기 전**)에 `np.array_equal(a, b)`는? — 값은 같았을까?

In [ ]:
e2_a = [1, 2, 3]   # 다른 상자를 고쳤으니 a는 그대로
e2_is = False      # 상자가 2개
e2_eq = True       # 복사 직후에는 값이 같다 (값이 같다 ≠ 같은 상자)

In [ ]:
# 확인
a = np.array([1, 2, 3])
b = a.copy()
eq_before = np.array_equal(a, b)
b[0] = 100
print('a =', a, '  a is b =', a is b, '  복사 직후 값이 같았나 =', eq_before)
check(e2_a, a, 'e2_a')
check(e2_is, a is b, 'e2_is')
check(e2_eq, eq_before, 'e2_eq')
print('통과!')

## E-3. 🔮 `-=` 와 `x = x - ...`
```python
a = np.array([5.0, 5.0])
b = a
b -= 1          # ①

c = a
c = c - 1       # ②
```
- `e3_after1`: ① 직후 `a`의 값
- `e3_after2`: ② 직후 `a`의 값 (①이 반영된 상태에서 시작)
- `e3_c_is_a`: ② 직후 `c is a`는?

In [ ]:
e3_after1 = [4.0, 4.0]   # -= 는 상자를 그 자리에서 고친다 → a도 4
e3_after2 = [4.0, 4.0]   # c - 1 은 새 상자를 만들고 이름표 c만 옮긴다 → a는 그대로 4
e3_c_is_a = False        # c는 이제 새 상자 [3, 3]에 붙어 있다

In [ ]:
# 확인
a = np.array([5.0, 5.0])
b = a
b -= 1
after1 = a.copy()
c = a
c = c - 1
print('① 뒤 a =', after1, '  ② 뒤 a =', a, '  c =', c, '  c is a =', c is a)
check(e3_after1, after1, 'e3_after1')
check(e3_after2, a, 'e3_after2')
check(e3_c_is_a, c is a, 'e3_c_is_a')
print('통과!')

## E-4. 🔮 숫자는 왜 괜찮았을까?
```python
n = 5
m = n
m -= 1
```
- `e4_n`: `n`의 값은?

> 정수·실수·문자열은 **고칠 수 없는 물건**(불변)이다. 그래서 `-=`도 그 자리에서 못 고치고, 새 숫자를 만들어 이름표를 옮긴다.

In [ ]:
e4_n = 5   # 숫자는 불변 → m -= 1 은 새 숫자 4를 만들고 이름표 m만 옮긴다

In [ ]:
# 확인
n = 5
m = n
m -= 1
print('n =', n, '  m =', m)
check(e4_n, n, 'e4_n')
print('통과! 이 함정은 배열·리스트·딕셔너리처럼 "고칠 수 있는 물건"에서만 드러난다.')

## E-5. 🔮 함수에 배열을 넘기면
함수 안의 매개변수도 **바깥 배열에 붙은 이름표**다.
```python
def sub_inplace(v):
    v -= 1

def sub_new(v):
    v = v - 1

a = np.array([10.0, 20.0])
sub_inplace(a)   # ①
sub_new(a)       # ②
```
- `e5_after1`: ① 직후 `a`
- `e5_after2`: ② 직후 `a`
- `e5_ret`: `sub_new(a)`가 돌려주는 값은? (`return`이 없다!) — `None`이면 문자열 `'None'`으로 적기

In [ ]:
e5_after1 = [9.0, 19.0]   # 함수 안의 v는 바깥 a와 같은 상자 → -= 로 직접 고침
e5_after2 = [9.0, 19.0]   # v = v - 1 은 함수 안에서만 새 상자. 바깥 a는 그대로
e5_ret = 'None'           # return이 없으면 None → 새 상자 결과는 버려진다

In [ ]:
# 확인
def sub_inplace(v):
    v -= 1

def sub_new(v):
    v = v - 1

a = np.array([10.0, 20.0])
sub_inplace(a)
after1 = a.copy()
ret = sub_new(a)
print('① 뒤 a =', after1, '  ② 뒤 a =', a, '  sub_new의 반환값 =', ret)
check(e5_after1, after1, 'e5_after1')
check(e5_after2, a, 'e5_after2')
check(e5_ret, str(ret), 'e5_ret')
print('통과! 바깥 배열을 바꾸지 않으려면 함수 안에서 .copy(), 새 결과를 쓰려면 return.')

## E-6. 🔮 리스트에 같은 상자를 여러 번 넣으면
그림 4-10의 `x_history`에서 나온 함정이다.
```python
x = np.array([0.0])
hist = []
for i in range(3):
    x += 1
    hist.append(x)          # copy 없이
```
- `e6_hist`: `hist`에 든 값 3개 (리스트로, 예: `[1, 2, 3]`)
- `e6_same`: `hist[0] is hist[2]`는?

In [ ]:
e6_hist = [3, 3, 3]   # 리스트에 '같은 상자'를 3번 넣었다 → 마지막 값만 3번 보인다
e6_same = True

In [ ]:
# 확인
x = np.array([0.0])
hist = []
for i in range(3):
    x += 1
    hist.append(x)
print('hist =', hist, '  hist[0] is hist[2] =', hist[0] is hist[2])
check(e6_hist, np.array(hist).ravel(), 'e6_hist')
check(e6_same, hist[0] is hist[2], 'e6_same')
print('통과! 그 순간의 값을 남기려면 hist.append(x.copy())')

## E-7. 🔮 슬라이스는 같은 상자, 리스트 인덱싱은 새 상자
넘파이에서 **슬라이스 `a[1:3]`는 원래 상자의 일부를 가리키는 창(뷰)**이고, **리스트로 꺼낸 `a[[1, 2]]`는 복사본**이다.
```python
a = np.arange(5)          # [0 1 2 3 4]
s = a[1:3]
s[0] = 99                 # ①

f = a[[3, 4]]
f[0] = -1                 # ②
```
- `e7_after1`: ① 직후 `a`
- `e7_after2`: ② 직후 `a`

In [ ]:
e7_after1 = [0, 99, 2, 3, 4]   # 슬라이스 = 원래 상자의 창 → a도 바뀐다
e7_after2 = [0, 99, 2, 3, 4]   # 리스트 인덱싱 = 복사본 → a는 그대로

In [ ]:
# 확인
a = np.arange(5)
s = a[1:3]
s[0] = 99
after1 = a.copy()
f = a[[3, 4]]
f[0] = -1
print('① 뒤 a =', after1, '  ② 뒤 a =', a, '  f =', f)
check(e7_after1, after1, 'e7_after1')
check(e7_after2, a, 'e7_after2')
print('통과!')

## E-8. 🔮 행 하나를 꺼내서 고치면 (`numerical_gradient_2d`와 연결)
```python
Mz = np.zeros((2, 3))
row = Mz[0]          # 숫자로 꺼낸 행 = 원래 상자의 창
row += 1
```
- `e8_M`: 이제 `Mz`의 값 (리스트 안에 리스트)

> `for idx, x in enumerate(X):`의 `x`도 이런 **창**이다. 그래서 `numerical_gradient`가 `x[i]`를 바꾸면 `X`가 바뀌고, 꼭 **복원**해야 한다.

In [ ]:
e8_M = [[1, 1, 1], [0, 0, 0]]   # row는 Mz 0번 행의 창 → 같은 칸을 고친다

In [ ]:
# 확인
Mz = np.zeros((2, 3))
row = Mz[0]
row += 1
print(Mz)
check(e8_M, Mz, 'e8_M')
print('통과!')

## E-9. ⭐🔮 딕셔너리로 가중치를 함께 쓸 때 (5장 미리보기)
5장에서는 신경망과 각 계층(Layer)이 **같은 가중치 상자**를 함께 쓴다. 그래서 갱신 방법에 따라 결과가 갈린다.
```python
params = {'W': np.ones(2)}
layer_W = params['W']            # 계층이 같은 상자를 들고 있음

params['W'] -= 0.5               # ① 그 자리에서 고치기
after1 = layer_W.copy()

params['W'] = params['W'] - 0.5  # ② 새 상자를 만들어 딕셔너리에 넣기
after2 = layer_W.copy()
```
- `e9_after1`: ① 직후 `layer_W`
- `e9_after2`: ② 직후 `layer_W`
- `e9_safe`: 계층이 **항상 최신 가중치**를 보게 하려면 ①, ② 중 어느 방식? (숫자)

In [ ]:
e9_after1 = [0.5, 0.5]   # 같은 상자를 고쳤으니 계층도 0.5를 본다
e9_after2 = [0.5, 0.5]   # 딕셔너리만 새 상자 [0, 0]으로 바뀌고, 계층은 옛 상자를 그대로 들고 있다
e9_safe = 1              # -= (그 자리에서 고치기) → 4장 5-4의 갱신 코드도 이 방식

In [ ]:
# 확인
params = {'W': np.ones(2)}
layer_W = params['W']
params['W'] -= 0.5
after1 = layer_W.copy()
params['W'] = params['W'] - 0.5
after2 = layer_W.copy()
print('① 뒤 layer_W =', after1, '  ② 뒤 layer_W =', after2, '  params[W] =', params['W'])
check(e9_after1, after1, 'e9_after1')
check(e9_after2, after2, 'e9_after2')
check(e9_safe, 1, 'e9_safe')
print('통과! 이름표 성질은 버그의 원인도 되고, 일부러 이용하기도 한다.')

## E-10. 🔧 고치기 — `init_x`를 지키는 경사 하강법
아래 함수는 결과는 맞지만 **넘겨준 `init_x`까지 바꿔 버린다**. 한 줄만 고쳐서 원본을 지키자.

In [ ]:
def safe_gradient_descent(f, init_x, lr=0.1, step_num=100):
    x = init_x.copy()          # 새 상자에서 시작 → init_x는 그대로
    for i in range(step_num):
        x -= lr * numerical_gradient(f, x)
    return x

In [ ]:
# 확인
start = np.array([-3.0, 4.0])
res = safe_gradient_descent(function_2, start)
check(res, np.array([0.0, 0.0]), '결과')
check(start, np.array([-3.0, 4.0]), 'start (바뀌면 안 됨)')
check(res is start, False, 'res is start')
print('통과!')

## E-11. 🔧 고치기 — 이동 기록이 전부 같은 값
아래 함수의 `history`가 전부 마지막 위치로만 채워진다. 고치자. (`x = init_x.copy()`는 이미 되어 있다)

In [ ]:
def path_of(f, init_x, lr=0.1, step_num=3):
    x = init_x.copy()
    history = []
    for i in range(step_num):
        history.append(x.copy())   # 그 순간의 값을 새 상자로 남기기
        x -= lr * numerical_gradient(f, x)
    return np.array(history)

In [ ]:
# 확인: x² 은 lr=0.1 이면 한 번에 0.8배
p = path_of(function_2, np.array([-3.0, 4.0]))
print(p)
check(p, np.array([[-3.0, 4.0], [-2.4, 3.2], [-1.92, 2.56]]), 'path')
print('통과!')

## E-12. 🔧 고치기 — 복원을 빼먹은 기울기
아래 함수는 **값 복원**이 빠져 있다. $f(x_0, x_1) = x_0 x_1$의 점 (3, 4)에서 기울기는 $(x_1, x_0) = (4, 3)$이어야 하는데...
1. 먼저 그대로 실행해서 무엇이 틀리는지 보고
2. 한 줄을 더해 고치자

In [ ]:
def gradient_no_restore(f, x):
    h = 1e-4
    grad = np.zeros_like(x)
    for i in range(x.size):
        tmp_val = x[i]
        x[i] = tmp_val + h
        fxh1 = f(x)
        x[i] = tmp_val - h
        fxh2 = f(x)
        grad[i] = (fxh1 - fxh2) / (2 * h)
        x[i] = tmp_val          # 복원: x는 바깥 배열과 같은 상자라서, 안 하면 바깥 값이 틀어진 채로 남는다
    return grad

In [ ]:
# 확인
f_xy = lambda x: x[0] * x[1]
pt = np.array([3.0, 4.0])
g = gradient_no_restore(f_xy, pt)
print('기울기 =', g, '  계산 뒤 pt =', pt)
check(pt, np.array([3.0, 4.0]), 'pt (바깥 배열이 그대로여야 함)')
check(g, np.array([4.0, 3.0]), '기울기')
print('통과!')

## E-13. 🔮 종합 — 한 줄씩 따라가기
```python
a = np.array([1.0, 2.0, 3.0])
b = a
c = a.copy()
d = a[0:2]
e = a[[0, 1]]

b[2] = 30
d[0] = 10
c[1] = 20
e[1] = 200
a = a * 2
```
마지막 줄까지 실행한 뒤의 값을 예측해 보자. (리스트)
- `e13_a`, `e13_b`, `e13_c`, `e13_d`

> 힌트: 마지막 줄 `a = a * 2`는 **새 상자**를 만들고 이름표 `a`만 옮긴다. `b`, `d`는 옛 상자를 계속 보고 있다.

In [ ]:
# 옛 상자: [1, 2, 3] → b[2]=30 → [1, 2, 30] → d[0]=10 → [10, 2, 30]  (b, d가 같은 상자를 고침)
e13_a = [20.0, 4.0, 60.0]   # 옛 상자 [10, 2, 30] × 2 → 새 상자
e13_b = [10.0, 2.0, 30.0]   # 옛 상자를 계속 봄
e13_c = [1.0, 20.0, 3.0]    # 처음에 복사해 둔 별도 상자
e13_d = [10.0, 2.0]         # 옛 상자의 0~1번 칸 창
# e = a[[0, 1]] 는 복사본이라 e[1] = 200 은 아무 데도 영향 없음

In [ ]:
# 확인 (이제 실행해서 맞춰 보기)
a = np.array([1.0, 2.0, 3.0])
b = a
c = a.copy()
d = a[0:2]
e = a[[0, 1]]
b[2] = 30
d[0] = 10
c[1] = 20
e[1] = 200
a = a * 2
print('a =', a, ' b =', b, ' c =', c, ' d =', d, ' e =', e)
check(e13_a, a, 'e13_a'); check(e13_b, b, 'e13_b')
check(e13_c, c, 'e13_c'); check(e13_d, d, 'e13_d')
print('통과! 이름표와 상자를 그릴 수 있으면 이 함정은 끝.')

# 끝!
연습 노트북: `ch04_review.ipynb`

**오늘 복습 한 줄씩**
- 숫자로 꺼내면 축이 사라지고, 리스트로 꺼내면 축이 남는다
- `y[행 목록, 열 목록]` = 같은 순서끼리 짝지어 꺼내기
- `meshgrid` → `flatten` → `np.array([X, Y]).T` = 점 하나가 행 하나
- 행렬의 한 행(열) = 그래프의 한 줄, 화살표는 `-grad`
- `=`는 이름표 붙이기(복사 X). 배열의 `-=`, `x[i] =`는 같은 상자를 고친다 → 원본을 지키려면 `.copy()`